<details>
  <summary><strong>MIT License（クリックで展開）</strong></summary>

  <pre>
MIT License
Copyright (c) 2023-2025 AI Course Notebook Contributors
Repository: https://github.com/arch-inform-kaken-group/ai-course-notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
  </pre>
</details>

# Phi-4-mini & DuckDuckGo検索API によるRAG（検索拡張生成）

今回のハンズオンは，検索拡張生成（Retrieval-Augmented Generation，RAG）による文章生成を体感していただくことを目標としています．RAGとは，検索エンジンなどを用いて外部知識を取得し，その情報を元に生成を行う手法です．

Phi Family は，Microsoftから提供されている小型の言語モデルです．オープンモデル（公開モデル）であるため，自由にダウンロード・利用することができます．

<br>

本ハンズオンで使用する **Phi-4-mini** は，その中でも小型でありながら高い性能を持つ言語モデルです．また，情報検索のために **DuckDuckGo API** を併用し，検索結果を用いて最新の情報や広範な情報源から回答を補強する **RAG** を構築します．


![Phi Models](https://raw.githubusercontent.com/microsoft/PhiCookBook/refs/heads/main/imgs/cover.png) <small>Phi Cookbook: Hands-On Examples with Microsoft's Phi Models (Jun. 10, 2025, 00:30 UTC). In GitHub. Retrieved from [https://github.com/microsoft/PhiCookBook](https://github.com/microsoft/PhiCookBook)</small>

<hr>

Phi Family モデル出典：

* ［Microsoft Blog（Phi-3）］[https://azure.microsoft.com/ja-jp/blog/introducing-phi-3-redefining-whats-possible-with-slms/](https://azure.microsoft.com/ja-jp/blog/introducing-phi-3-redefining-whats-possible-with-slms/)
* ［Microsoft Blog（Phi-4）］[https://azure.microsoft.com/en-us/blog/one-year-of-phi-small-language-models-making-big-leaps-in-ai/](https://azure.microsoft.com/en-us/blog/one-year-of-phi-small-language-models-making-big-leaps-in-ai/)
* ［arXiv（Phi-3）］[Abdin, Marah et al. “Phi-3 Technical Report: A Highly Capable Language Model Locally on Your Phone.” *arXiv* (2024)](https://arxiv.org/abs/2404.14219)
* ［arXiv（Phi-4）］[Abdin, Aneja et al. “Phi-4 Technical Report” *arXiv* (2024)](https://arxiv.org/abs/2412.08905)
* ［GitHub］[https://github.com/microsoft/PhiCookBook](https://github.com/microsoft/PhiCookBook)
* ［HuggingFace (Phi-3-mini)］[https://huggingface.co/microsoft/Phi-3-mini-4k-instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct)
* ［HuggingFace (Phi-4-mini)］[https://huggingface.co/microsoft/Phi-4-mini-instruct](https://huggingface.co/microsoft/Phi-4-mini-instruct)




#  検索拡張生成（RAG）とは

検索拡張生成（Retrieval-Augmented Generation，RAG）とは，大規模言語モデル（LLM）などに対して，外部の知識情報（検索結果など）を与えることで，言語モデルの訓練時に学習していなかったデータへも対応する技術です．


![RAG](https://python.langchain.com/assets/images/rag_retrieval_generation-1046a4668d6bb08786ef73c56d4f228a.png
)
<br>
<small>Build a Retrieval Augmented Generation (RAG) App: Part 1 (Jul. 7, 2025, 00:30 UTC). In LangChain. Retrieved from [https://python.langchain.com/docs/tutorials/rag/](https://python.langchain.com/docs/tutorials/rag/)</small>

<br>

RAGは大まかに以下の流れで構成されます：

1. クエリ生成（Query）
　ユーザの質問や指示を元に，検索エンジンに投げるためのクエリが作成する

2. 情報検索（Retrieval）
　外部の検索システム（今回は DuckDuckGo API）を用いて，関連する情報を取得する

3. 文章生成（Generation）
　取得した情報（今回は **スニペット**）を参照情報として，LLMが回答を生成する


RAGは「検索＋生成」を組み合わせることで，LLM単体では保持していない知識や最新の情報を活用します．

<br>

<hr>
注）スニペットとは，検索結果に表示されるWebページのタイトル下にある短い説明文のことです．本来のRAGでは，検索結果に含まれるURL先のページを取得し，本文中の情報をベクトル化して利用するのが一般的です．しかし，本ハンズオンでは簡易化のため，検索結果の「スニペット（＝断片的な説明文）」のみを使用しています．

<br>











## **Limitations**（講義利用におけるモデル/検索APIの制約・精度の限界）

本ハンズオンで使用するモデル：Phi-4-miniは，ChatGPT等の非公開で大規模な言語モデルと比較すると，**精度は非常に悪い** です．また，日本語の入出力は可能ですが，あまり流暢ではありません．複雑な指示や曖昧な表現に対する対応力にも限界があります．

<br>

また，本ハンズオンで使用する検索API：DuckDuckGOも，普段皆さんがお使いの Google検索 などと比較すると，**検索精度は悪い** です．検索キーワードに対し，必ずしも関連性が高いWebページが上位に表示されるとは限りません。
更に，本ハンズオンでは実装を簡易化するため，検索結果のサイトへは遷移せず，検索時に表示される **スニペット** を情報源として使用します．スニペットに必要な情報が含まれていない場合や，誤情報や古い情報が含まれている場合などは，生成される文章にも誤りが生じます．

<br>

しかし，以下の条件をすべて満たすことを優先してこの構成を採用しています：

1. **無償で簡易に（支払い情報などの登録なしに）** 利用可能な文章生成モデルや検索APIであること
2. 文章生成モデルと検索APIの **パラメータや構成要素が自由に操作可能** であること
3. **学習用** に，検索と生成の流れを明示的に確認できること
4. 無料版Colabの環境でも実行可能な **小型のオープンモデル** であること

<br>

例えば，ChatGPT（ブラウザ版）は， 2 と 3 の条件を満たしません．具体的には，2025年7月現在のブラウザ版のChatGPTは，「ウェブで検索する」を選択せずとも，その入力文が【Web検索するべき文章か否か】を自動識別しています．つまり，検索と生成の流れを明示的に確認することが難しくなっています．

<br>

他，ChatGPT（API版）は 1 と 4 の条件を満たさない，Google検索（API版）は支払い情報などの登録が必要，など授業で使用するにあたり様々な制約があります．

<br>

本ハンズオンの主目的は，
* 文章生成モデルで設定可能なパラメータを確認すること
* **RAGという技術の仕組みを体験すること**

です．

<br>文章生成モデルや検索APIの性能自体ではなく，文章生成モデルにおけるパラメータの効果や，**検索と文章生成モデルの連携** に着目してください．

<br>




## ライブラリインストール・インポート

In [ ]:
# 実行時間：1～2分

# ライブラリインストール
# duckduckgo-search: API キー不要のウェブ検索ライブラリ
# https://pypi.org/project/duckduckgo-search/8.1.0/
# accelerate：PyTorchでの実装を一部簡易化するライブラリ
!pip install duckduckgo-search==8.1.0

# ライブラリインポート

# IPython.display : Colabなどのipython環境における表示機能を持つライブラリ
# 　display:一部のデータをprint()よりも綺麗に出力する
# 　Markdown:マークダウン記法のテキストで格納されたデータを表示する
from IPython.display import display, Markdown
from PIL import Image # 画像処理用ライブラリの一つ
import requests # HTTP通信用ライブラリの一つ
from io import BytesIO #io:標準ライブラリ，入出力を扱う
# transformers:Huggingface社が公開しているtransformer派生モデルを実装するフレームワーク
import transformers
import random # random：標準ライブラリ，擬似乱数発生器
import numpy as np # numpy：数値計算用ライブラリの一つ
import torch # pyTorch：深層学習ライブラリの一つ
from duckduckgo_search import DDGS

# NVIDIA GPU環境の有無を確認する
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.6 MB/s eta 0:00:00
Thu Jul 10 04:50:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                    

## 擬似乱数seed固定（pyTorch版）

In [ ]:
def torch_set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    # Pytorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms = True


torch_set_seed(seed=0)

## Phi-4-mini モデルのダウンロードと準備

In [ ]:
# 実行時間：10～12分

model_id ="microsoft/Phi-4-mini-instruct"

# メモリ開放用コード
if ("generator" in locals())or("generator" in globals()):
    import gc
    generator.model.cpu()
    del generator, processor
    gc.collect()
    torch.cuda.empty_cache()


# 今回はトークン化の処理を自動で行う機能(pipeline）を使用します：

generator = transformers.pipeline(
    task = "text-generation",
    model = model_id,
    device_map = "cuda",
    torch_dtype = "auto",
)

# Phi-4-mini用の入力へ対応させるための前処理設定を読み込む
processor = transformers.AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

# 読み込んだ事前学習・指示応答調整済モデルと，このモデル用の「トークン」情報が出力される
processor = transformers.AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
display(processor)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Device set to use cuda


configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


GPT2TokenizerFast(name_or_path='microsoft/Phi-4-mini-instruct', vocab_size=200019, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	199999: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200018: AddedToken("<|endofprompt|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200019: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	200020: AddedToken("<|end|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	200021: AddedToken("<|user|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	200022: AddedToken("<|system|>", rstrip=True, lstrip=False, s

##  ① 文章生成モデルにおける生成時のパラメータ

ここでは，特に使われることが多い **temperature（温度）** と **max_new_tokens** について扱います．

他のパラメータも詳しく知りたい方はこちら：

* [生成パラメータの戦略] https://huggingface.co/docs/transformers/ja/generation_strategies
<br>
* [生成パラメータの一覧] https://huggingface.co/docs/transformers/v4.53.1/ja/main_classes/text_generation#transformers.GenerationConfig

<hr>
<small>(注）"repetition_penalty" はモード崩壊を軽減するために入れています．本ハンズオンや課題では詳しく扱いません．"do_sample" はFalse（無効）に設定すると，同じ入力文に対しては，常に同じ出力となります．本ハンズオンでは，temperatureパラメータの効果を確認するため，"do_sample" はTrue（有効）に設定しています．</small>


### モデルへの入力形式を確認する

In [ ]:
# 文章の問い合わせ文を，プロンプト（チャット様式版）へ変換

messages = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content": "ドラゴンフルーツの美味しい食べ方を3つ教えてください。"},
]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
display(prompt)

'<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。<|end|><|user|>ドラゴンフルーツの美味しい食べ方を3つ教えてください。<|end|><|assistant|>'

### モデルからの出力形式を確認する

In [ ]:
# 推論時のパラメータ
generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

outputs = generator(prompt, **generate_kwargs)

print("モデルからの出力（そのまま）：\n")
display(outputs)

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

print("<|assistant|>以降を切り出し，かつマークダウン書式であると処理して表示：\n")
display(Markdown(outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()))

モデルからの出力（そのまま）：



[{'generated_text': '<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。<|end|><|user|>ドラゴンフルーツの美味しい食べ方を3つ教えてください。<|end|><|assistant|>1. ドラゴンフルーツスムージー: ドラゴンフルーツ、バナナ、ライムジュース、氷、水を混ぜて冷やしながら飲む。\n\n2. ドラゴンフルーツパイナップルサラダ: ドラゴンフルーツ、赤ピーマン、カリフラワー、レモン汁、オリーブオイルでドレッシングしたものを混ぜる。\n\n3. ドラゴンフルーツチーズケーキ: ドラゴンフルーツジャムとクリームチーズを使用して作ったケーキを焼き、上に軽くコーティングする。'}]


------------------------------------------------------------------------------------------------------------------------------------------------------

<|assistant|>以降を切り出し，かつマークダウン書式であると処理して表示：



1. ドラゴンフルーツスムージー: ドラゴンフルーツ、バナナ、ライムジュース、氷、水を混ぜて冷やしながら飲む。

2. ドラゴンフルーツパイナップルサラダ: ドラゴンフルーツ、赤ピーマン、カリフラワー、レモン汁、オリーブオイルでドレッシングしたものを混ぜる。

3. ドラゴンフルーツチーズケーキ: ドラゴンフルーツジャムとクリームチーズを使用して作ったケーキを焼き、上に軽くコーティングする。

### temperature の値を大きく（0.000001 ⇒ 1.0）すると...

In [ ]:
generate_kwargs = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

outputs = generator(prompt, **generate_kwargs)

print("モデルからの出力（そのまま）：\n")
display(outputs)

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

print("<|assistant|>以降を切り出し，かつマークダウン書式であると処理して表示：\n")
display(Markdown(outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()))

モデルからの出力（そのまま）：



[{'generated_text': '<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。<|end|><|user|>ドラゴンフルーツの美味しい食べ方を3つ教えてください。<|end|><|assistant|>1. トーストまたはチューインング: 皮と内臓を外し、皮を切り取り、お好みで塩こだましてパンに焼いて、バターやナッツなどを添えていただけます。\n\n2. ジャム：レーズン、キャンディボーン、スパイスを混ぜ合わせて準備した後、ドラゴンフルーツのジュースを使って自然にジャムを作ることができます。サーブする前に冷却して固めましょう。\n\n3. ドリンク：新鮮な葉を使ったドライブスルーシードラゴンフルーツやドラゴンフルーツマグカップ、アイスクリームのトッピングとして使用します。果物の中に緑ティーやレモネードを入れてお喝采を送ります。'}]


------------------------------------------------------------------------------------------------------------------------------------------------------

<|assistant|>以降を切り出し，かつマークダウン書式であると処理して表示：



1. トーストまたはチューインング: 皮と内臓を外し、皮を切り取り、お好みで塩こだましてパンに焼いて、バターやナッツなどを添えていただけます。

2. ジャム：レーズン、キャンディボーン、スパイスを混ぜ合わせて準備した後、ドラゴンフルーツのジュースを使って自然にジャムを作ることができます。サーブする前に冷却して固めましょう。

3. ドリンク：新鮮な葉を使ったドライブスルーシードラゴンフルーツやドラゴンフルーツマグカップ、アイスクリームのトッピングとして使用します。果物の中に緑ティーやレモネードを入れてお喝采を送ります。

### 例文「私は大学祭で実行委員をしています。何か面白そうな催し物の案はありますか？」

In [ ]:
messages = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content": "私は大学祭で実行委員をしています。何か面白そうな催し物の案はありますか？"},
]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

generate_kwargs_high_temperature = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

generate_kwargs_few_tokens = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 16,
    "repetition_penalty": 1.05
}

generate_kwargs_high_temperature_and_few_tokens = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 16,
    "repetition_penalty": 1.05
}



print()
print(f"temperature：{generate_kwargs['temperature']}，max_new_tokens：{generate_kwargs['max_new_tokens']}")
outputs = generator(prompt, **generate_kwargs)
display(Markdown(outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_high_temperature['temperature']}，max_new_tokens：{generate_kwargs_high_temperature['max_new_tokens']}")
outputs_high_temperature = generator(prompt, **generate_kwargs_high_temperature)
display(Markdown(outputs_high_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_few_tokens['temperature']}，max_new_tokens：{generate_kwargs_few_tokens['max_new_tokens']}")
outputs_few_tokens = generator(prompt, **generate_kwargs_few_tokens)
display(Markdown(outputs_few_tokens[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_high_temperature_and_few_tokens['temperature']}，max_new_tokens：{generate_kwargs_high_temperature_and_few_tokens['max_new_tokens']}")
outputs_high_temperature_and_few_tokens = generator(prompt, **generate_kwargs_high_temperature_and_few_tokens)
display(Markdown(outputs_high_temperature_and_few_tokens[0]["generated_text"].split("<|assistant|>")[-1].strip()))


temperature：1e-06，max_new_tokens：512


もちろん！以下はいくつかのアイデアです：

1. 文化的なパレード：異なる国や地域を表すグループが伝統的な衣装、ダンス、料理を披露します。
2. 学生コンテスト：スポーツ、演劇、科学などのさまざまなカテゴリーで競う学生コンテストを開催できます。
3. ビンゴゲーム：大きなビンゴボードを設置して、参加者がチケットを購入できるようにします。最終的な勝者には賞品があります。
4. フードフェスティバル：各地元の食べ物を提供するフランチャイズスタンドを設置します。参加者は自分のお気に入りの料理を試してみることができます。
5. 音楽とダンスのショー：学生たちがパフォーマンスを行い、観客を楽しませます。
6. スポーツトーナメント：サッカー、バスケットボール、野球などのスポーツイベントを組織します。
7. DIY工芸市場：学生が自分の作品を販売または展示するためのスペースを提供します。
8. コミュニケーションワークショップ：コミュニケーションスキルを向上させるためのワークショップやセッションを開催します。
9. パーティーゲーム：クラシックなパーティーゲーム（モンタージュ、ペナルティ・イン・ザ・ハウス、プロジェクト・マスターズ）を組織します。
10. メディアブース：学生ジャーナリストや写真家が最新ニュースやイベントをカバーし、視聴者にインタラクティブな体験を提供します。

これらのアイデアの中から、あなたの大学祭の雰囲気や興味に合ったものを選んでください。祝祭を成功させてください！


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1.0，max_new_tokens：512


はい、以下はいくつかの魅力的な提案です：

1. コメディバーストレート: 異なるエリアを横断して来るミュージカルコメディグループを招待します。
2. バイトフェスティバル: 学生がピクニックデザインバリエーションを作り、それぞれの料理とバンドパフォーマンスにマイクロブログ投稿を添えて展示します。
3. スポーツダブルダウン: 形式あて本や伝統的なスポーツイベント（フットサル、ローラースケートなど）を組み合わせます。
4. クイーンズコンテスト: 多様なクラフトステーションを設置して、テーマ別のパネル・ラウンドで学生を招き入れることに興奮します。
5. モダンなダンスフライナップショー: 学校の学生と地元ダンサーを出演させ、ポップ、ヒップホップ、その他のダンススタイルを披露できます。

どのアイデアも異なる参加者を引き込むためのでしょう。イベントで共有したい興味や活動を教えてください – よらしい提案を提供することができます！


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1e-06，max_new_tokens：16


もちろん！以下はいくつかのアイデアです：

1.


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1.0，max_new_tokens：16


もちろん！「スマートコーナー」プロジェクトを試してみ

temperatureパラメータを変えて3回繰り返し実行してみます．

In [ ]:
messages = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content": "私は大学祭で実行委員をしています。何か面白そうな催し物の案はありますか？"},
]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


generate_kwargs_low_temperature = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}
generate_kwargs_medium_temperature = {
    "do_sample": True,
    "temperature": 0.7,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05

}
generate_kwargs_high_temperature = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}

for n in range(1,4):
    outputs_low_temperature = generator(prompt, **generate_kwargs_low_temperature)
    print(f"\ntemperature：{generate_kwargs_low_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_low_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

for n in range(1,4):
    outputs_medium_temperature = generator(prompt, **generate_kwargs_medium_temperature)
    print(f"\ntemperature：{generate_kwargs_medium_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_medium_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

for n in range(1,4):
    outputs_high_temperature = generator(prompt, **generate_kwargs_high_temperature)
    print(f"\ntemperature：{generate_kwargs_high_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_high_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))



temperature：1e-06 の実行1回目：


もちろん！以下はいくつかのアイデアです：

1. 文化的なパレード：異なる国や地域を表すグループが伝統的な衣装、ダンス、料理を披露します。
2. 学生コンテスト：スポーツ、演劇、科学などのさまざまなカテゴリーで競う学生コンテストを開催できます。
3. ビンゴゲーム：大きなビンゴボードを設置して、参加者がチケットを購入できるようにします。
4. フードフェスティバル：


temperature：1e-06 の実行2回目：


もちろん！以下はいくつかのアイデアです：

1. 文化的なパレード：異なる国や地域を表す参加者が伝統的な衣装、ダンス、料理を披露します。
2. 学生コンテスト：歌唱、演劇、科学プロジェクトなどのさまざまなカテゴリーで競争することができます。
3. スポーツトーナメント：フットサル、バスケットボール、野球などのチームスポーツを組織してみてください。
4. ビジュアルエフェ


temperature：1e-06 の実行3回目：


...

はい、特定の趣味やカーストを持つバレンタインデーのクラフトブリック大会と「最小のコンクール」ボランティアが参加できます。「スパークサンプリング・シンフォニー」という名前のピアノナイトや「マリンミューズ・ウォーターフロントフェスティバル」という名前のビーチハウス会もあり、日本の伝統的な芸能の演技コンペを含め、文化的な雰囲気を作り出すことができるかもし


temperature：1.0 の実行2回目：


もちろん！次の活動を試してみてください：

**グルメビッグクイズ**

1. **準備：**
   - 多様な食べ物やその起源に関する人気質問を集めます（例：日本の寿司に関する質問）。
   - 長方形状の地図アイテムを購入して、それぞれが異なる国ごとに飾り付けします。

2. **実行：**
   - クイズを解くための時間と休憩を決定します。
   - 地図用にキャンバス


temperature：1.0 の実行3回目：


もちろん！コメディパフォーマンスと呼ばれるものを組み入れてみてください。これは参加者が自分で観客に読み上げる即興のユーモアのセッションです：

1. まず、選挙を広めます - 志願する人たちに、自分の冗談やジョークを書いてもらい、それぞれが短い（約2-3分）の即興スケッチだと言ってもいいです。
2. ステージが設置される準備をします。
3. 劇

### 例文「私は大学祭で実行委員をしています。学生の興味を引けそうなキャッチフレーズの案はありますか？」

In [ ]:
messages = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content": "私は大学祭で実行委員をしています。学生の興味を引けそうなキャッチフレーズの案はありますか？"},
]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

generate_kwargs_high_temperature = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

generate_kwargs_few_tokens = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 16,
    "repetition_penalty": 1.05
}

generate_kwargs_high_temperature_and_few_tokens = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 16,
    "repetition_penalty": 1.05
}



print()
print(f"temperature：{generate_kwargs['temperature']}，max_new_tokens：{generate_kwargs['max_new_tokens']}")
outputs = generator(prompt, **generate_kwargs)
display(Markdown(outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_high_temperature['temperature']}，max_new_tokens：{generate_kwargs_high_temperature['max_new_tokens']}")
outputs_high_temperature = generator(prompt, **generate_kwargs_high_temperature)
display(Markdown(outputs_high_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_few_tokens['temperature']}，max_new_tokens：{generate_kwargs_few_tokens['max_new_tokens']}")
outputs_few_tokens = generator(prompt, **generate_kwargs_few_tokens)
display(Markdown(outputs_few_tokens[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_high_temperature_and_few_tokens['temperature']}，max_new_tokens：{generate_kwargs_high_temperature_and_few_tokens['max_new_tokens']}")
outputs_high_temperature_and_few_tokens = generator(prompt, **generate_kwargs_high_temperature_and_few_tokens)
display(Markdown(outputs_high_temperature_and_few_tokens[0]["generated_text"].split("<|assistant|>")[-1].strip()))


temperature：1e-06，max_new_tokens：512


"学び、笑い、つながり！ #UniFestVibes"


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1.0，max_new_tokens：512


"🎉✨ 学問の集まれ！Discover, Connect, Celebrate University Festival! 🎓"


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1e-06，max_new_tokens：16


"学び、笑い、つながり - 大学祭の冒


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1.0，max_new_tokens：16


"変革力: 変革社会へ！" "学生一人

temperatureパラメータを変えて3回繰り返し実行してみます．

In [ ]:
messages = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content": "私は大学祭で実行委員をしています。学生の興味を引けそうなキャッチフレーズの案はありますか？"},
]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


generate_kwargs_low_temperature = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}
generate_kwargs_medium_temperature = {
    "do_sample": True,
    "temperature": 0.7,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05

}
generate_kwargs_high_temperature = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}

for n in range(1,4):
    outputs_low_temperature = generator(prompt, **generate_kwargs_low_temperature)
    print(f"\ntemperature：{generate_kwargs_low_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_low_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

for n in range(1,4):
    outputs_medium_temperature = generator(prompt, **generate_kwargs_medium_temperature)
    print(f"\ntemperature：{generate_kwargs_medium_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_medium_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

for n in range(1,4):
    outputs_high_temperature = generator(prompt, **generate_kwargs_high_temperature)
    print(f"\ntemperature：{generate_kwargs_high_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_high_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))



temperature：1e-06 の実行1回目：


"学び、笑い、つながり！ #UniFestVibes"


temperature：1e-06 の実行2回目：


"学び、笑い、つながり - 大学祭の冒険が始まる！"


temperature：1e-06 の実行3回目：


...

"スーパーヒーロー、グルーチョ、トイと一緒: エキサイティングな大学祭への皆さん！さあ、大いなる試合を見よう!"


temperature：1.0 の実行2回目：


"スカイラップ: 新しい学問の冒険、アイデアと笑いが一緒になる; 参加し、貢献しよう！"


temperature：1.0 の実行3回目：


"学びと友情の中で、Celebrate Higher Unity!"

### 例文「新潟県、長野県、山梨県の県庁所在地はどこですか？」

In [ ]:
messages = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content": "新潟県、長野県、山梨県の県庁所在地はどこですか？"},
]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

generate_kwargs_high_temperature = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05
}

generate_kwargs_few_tokens = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 16,
    "repetition_penalty": 1.05
}

generate_kwargs_high_temperature_and_few_tokens = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 16,
    "repetition_penalty": 1.05
}



print()
print(f"temperature：{generate_kwargs['temperature']}，max_new_tokens：{generate_kwargs['max_new_tokens']}")
outputs = generator(prompt, **generate_kwargs)
display(Markdown(outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_high_temperature['temperature']}，max_new_tokens：{generate_kwargs_high_temperature['max_new_tokens']}")
outputs_high_temperature = generator(prompt, **generate_kwargs_high_temperature)
display(Markdown(outputs_high_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_few_tokens['temperature']}，max_new_tokens：{generate_kwargs_few_tokens['max_new_tokens']}")
outputs_few_tokens = generator(prompt, **generate_kwargs_few_tokens)
display(Markdown(outputs_few_tokens[0]["generated_text"].split("<|assistant|>")[-1].strip()))
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


print(f"temperature：{generate_kwargs_high_temperature_and_few_tokens['temperature']}，max_new_tokens：{generate_kwargs_high_temperature_and_few_tokens['max_new_tokens']}")
outputs_high_temperature_and_few_tokens = generator(prompt, **generate_kwargs_high_temperature_and_few_tokens)
display(Markdown(outputs_high_temperature_and_few_tokens[0]["generated_text"].split("<|assistant|>")[-1].strip()))


temperature：1e-06，max_new_tokens：512


新潟県：新潟市  
長野県：長野市  
山梨県：甲府市


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1.0，max_new_tokens：512


新潟県庁：新潟市  
長野県庁：長野市  
山梨県庁：山梨都市科教館前（旧宮沢庁舎）


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1e-06，max_new_tokens：16


新潟県：新潟市  
長野県：長野


------------------------------------------------------------------------------------------------------------------------------------------------------

temperature：1.0，max_new_tokens：16


新潟県: 新潟市  
長野県: 長野

temperatureパラメータを変えて3回繰り返し実行してみます．

In [ ]:
messages = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content": "新潟県、長野県、山梨県の県庁所在地はどこですか？"},
]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


generate_kwargs_low_temperature = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}
generate_kwargs_medium_temperature = {
    "do_sample": True,
    "temperature": 0.7,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05

}
generate_kwargs_high_temperature = {
    "do_sample": True,
    "temperature": 1.0,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}

for n in range(1,4):
    outputs_low_temperature = generator(prompt, **generate_kwargs_low_temperature)
    print(f"\ntemperature：{generate_kwargs_low_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_low_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

for n in range(1,4):
    outputs_medium_temperature = generator(prompt, **generate_kwargs_medium_temperature)
    print(f"\ntemperature：{generate_kwargs_medium_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_medium_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

for n in range(1,4):
    outputs_high_temperature = generator(prompt, **generate_kwargs_high_temperature)
    print(f"\ntemperature：{generate_kwargs_high_temperature['temperature']} の実行{n}回目：")
    display(Markdown(outputs_high_temperature[0]["generated_text"].split("<|assistant|>")[-1].strip()))



temperature：1e-06 の実行1回目：


新潟県：新潟市  
長野県：長野市  
山梨県：甲府市


temperature：1e-06 の実行2回目：


新潟県：新潟市  
長野県：長野市  
山梨県：甲府市


temperature：1e-06 の実行3回目：


...

新潟：新潟市、長野：長野市、山梨：甲府市


temperature：1.0 の実行2回目：


新潟県は新潟市、新井川本町〒100-8570 新潟県新潟市中央区3-5-1にある、新潟市が県庁所在地です。長野県は長野市にあります。山梨県の県庁所在地は甲府市でございます。


temperature：1.0 の実行3回目：


新潟：新潟市  
長野：長野市  
山梨：甲州市

## ② 検索拡張生成（RAG）の効果

### 非RAG（通常の生成）時の出力を確認する；例文「日本の人口は？」

In [ ]:
query = "日本の人口は？"

messages_non_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content":f"{query}"},
    ]
prompt_non_rag = processor.apply_chat_template(messages_non_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト：")
display(prompt_non_rag)
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}

outputs_non_rag = generator(prompt_non_rag, **generate_kwargs)
print("非RAG（通常の生成）時の出力：")
display(Markdown(outputs_non_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))


モデルへの入力プロンプト：


'<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。<|end|><|user|>日本の人口は？<|end|><|assistant|>'


------------------------------------------------------------------------------------------------------------------------------------------------------

非RAG（通常の生成）時の出力：


2021年時点で、日本の推定人口は約125.8百万人です。ただし、最新の正確な数字を得るには、国勢調査や信頼できる統計データを確認してください。

### RAG時の出力を確認する；例文「日本の人口は？」

In [ ]:
query = "日本の人口は？"
k = 5

# DuckDuckGo 検索APIの検索結果を取得し
# 上位 k 件のスニペットをプロンプトに含めて回答を生成

ddgs = DDGS(headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=20)
results = list(ddgs.text(query, max_results=k,  region='ja-jp', safesearch='moderate',
                         timelimit='y', backend='bing'
                         ))

# スニペットをプロンプト用に整形
context = "\n\n".join(
    f"Title: {item['title']}\nSnippet: {item['body']}"
    for item in results
)

messages_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。"},
    {"role": "user", "content":f"{query}\n\n参考情報:{context}"},
]

prompt_rag = processor.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト：")
print(prompt_rag)
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 128,
    "repetition_penalty": 1.05
}

outputs_rag = generator(prompt_rag, **generate_kwargs)
print("RAG時の出力：")
display(Markdown(outputs_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))


モデルへの入力プロンプト：
<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。<|end|><|user|>日本の人口は？

参考情報:Title: 日本の人口は1億2488万人余 去年より約53万人減 2024年1月1 ...
Snippet: 2024年7月24日 · 総務省のまとめによりますと、2024年1月1日現在の住民基本台帳をもとにした外国人を含めた日本の総人口は、1億2488万5175人でした。 前の年の同じ時期と比べて53 …

Title: 日本の総人口 推計1億2380万人余 14年連続減 日本人の減少最大
Snippet: 2025年4月14日 · 総務省は2024年10月1日現在の人口推計を発表し、外国人を含めた日本の総人口は1億2380万2000人で、前の年よりも55万人、率にして0.44％減りました。

Title: グラフで見る日本の人口推移(過去と未来・将来の推測まで)と ...
Snippet: 4 日前 · 下記のバーチャートレースは、日本の総人口の世界順位の変遷です。 過去の1960年から2024年までの日本の世界ランキングの全履歴 を、バーチャートレースにてグラフで見える …

Title: 日本の総人口14年連続で減少、1億2380万2000人…75歳以上 ...
Snippet: 2025年4月14日 · 総務省は14日、2024年10月1日時点の日本の総人口推計（外国人含む）を発表した。 前年比55万人（0・44％）減の1億2380万2000人で、14年連続で減少した。 出生者 …

Title: 2025年最新情報 日本総人口は1億2359万人へ減少、前年比56 ...
Snippet: 2025年1月21日 · 2024年8月1日時点での日本人人口は1億2051万8千人で、前年同月に比べ89万2千人（0.73％）減少しました。 これは全体の減少幅をさらに押し上げる要因となっていま …<|end|><|assistant|>

-------------------------------------------------------------------------------------------------------------

2024年1月1日のデータに基づくと、日本の総人口は1億2488万5175人でした。これには外国人も含まれています。

### 例文「日本の首相は？」

In [ ]:
query = "日本の首相は？"
k = 10

# DuckDuckGo 検索APIの検索結果を取得し
# 上位 k 件のスニペットをプロンプトに含めて回答を生成

ddgs = DDGS(headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=20)
results = list(ddgs.text(query, max_results=k,  region='ja-jp', safesearch='moderate', timelimit='y', backend='bing'))

# スニペットをプロンプト用に整形
context = "\n\n".join(
    f"Title: {item['title']}\nSnippet: {item['body']}"
    for item in results
)


messages_non_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content":f"{query}"},
    ]

messages_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。"},
    {"role": "user", "content":f"{query}\n\n参考情報:{context}"},
]

prompt_non_rag = processor.apply_chat_template(messages_non_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト（通常の生成）：\n")
display(prompt_non_rag)
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")



prompt_rag = processor.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト（RAG）：\n")
print(prompt_rag)
print("\n"+ "".join([ "=" for n in range(150)]) + "\n")

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 256,
    "repetition_penalty": 1.05
}

outputs_non_rag = generator(prompt_non_rag, **generate_kwargs)
print("非RAG（通常の生成）時の出力：")
display(Markdown(outputs_non_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

outputs_rag = generator(prompt_rag, **generate_kwargs)
print("RAG時の出力：")
display(Markdown(outputs_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))


モデルへの入力プロンプト（通常の生成）：



'<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。<|end|><|user|>日本の首相は？<|end|><|assistant|>'


------------------------------------------------------------------------------------------------------------------------------------------------------

モデルへの入力プロンプト（RAG）：

<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。<|end|><|user|>日本の首相は？

参考情報:Title: 内閣総理大臣の一覧 - Wikipedia
Snippet: 2025年7月3日 · 内閣総理大臣の一覧 （ないかくそうりだいじんのいちらん）は、 日本 の 行政府の長 または 内閣 の首長である 内閣総理大臣 を務めた人物の一覧である。

Title: 【一覧】石破新内閣の顔ぶれ 閣僚プロフィールと内閣のデータ ...
Snippet: 2024年10月1日 · 石破内閣は、皇居での総理大臣の親任式と閣僚の認証式を経て、正式に発足しました。 閣僚の顔ぶれの一覧です。 村上誠一郎氏は衆議院愛媛2区選出の当選12回で、72 …

Title: 内閣総理大臣の決め方は？指名の流れや誰がする？いつ変わる ...
Snippet: 2025年6月4日 · 日本の総理大臣は、国民が直接選ぶのではなく、国会議員の中から選ばれます。 通常、国会で最も多くの議席を持つ政党の代表が就任します。

Title: トランプ大統領“日本に25％の関税” 石破首相の受け止めは ...
Snippet: 日本には来月1日から25％の関税を課すとした、トランプ大統領の書簡を石破首相はどう受け止めたのでしょうか？ 中継です。

Title: 内閣総理大臣の指名-令和6年10月1日 - 政府広報オンライン
Snippet: 令和6年10月1日午後、衆参両院にて首相指名投票が行われ、石破茂議員が、伊藤博文初代内閣総理大臣から数えて第102代目の内閣総理大臣として指名されました。

Title: 日本の首相は「コロコロ変わって弱い」から「一強」へ ...
Snippet: 2024年12月4日 · 政治学者で上智大学国際教養学部教授を務める中野晃一氏は、自身が監修した

現在の日本の首相は岸田文雄です。彼は2021年10月4日に就任しました。最新情報を確認するためには、信頼できるニュースソースを参照してください。


------------------------------------------------------------------------------------------------------------------------------------------------------

RAG時の出力：


現在の日本の首相は石破茂です。彼は令和6年10月1日に首相に指名されました。

### 例文「新潟県へ旅行に行きます。おすすめを教えてください。」

In [ ]:

query = "新潟県へ旅行に行きます。おすすめを教えてください。"
k = 5

# DuckDuckGo 検索APIの検索結果を取得し
# 上位 k 件のスニペットをプロンプトに含めて回答を生成

ddgs = DDGS(headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=20)
results = list(ddgs.text(query, max_results=k,  region='ja-jp', safesearch='moderate', timelimit='y', backend='bing'))

# スニペットをプロンプト用に整形
context = "\n\n".join(
    f"Title: {item['title']}\nSnippet: {item['body']}"
    for item in results
)


messages_non_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content":f"{query}"},
    ]

messages_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。"},
    {"role": "user", "content":f"{query}\n\n参考情報:{context}"},
]

prompt_non_rag = processor.apply_chat_template(messages_non_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト（通常の生成）：\n")
display(prompt_non_rag)
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")



prompt_rag = processor.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト（RAG）：\n")
print(prompt_rag)
print("\n"+ "".join([ "=" for n in range(150)]) + "\n")

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 256,
    "repetition_penalty": 1.05
}

outputs_non_rag = generator(prompt_non_rag, **generate_kwargs)
print("非RAG（通常の生成）時の出力：")
display(Markdown(outputs_non_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

outputs_rag = generator(prompt_rag, **generate_kwargs)
print("RAG時の出力：")
display(Markdown(outputs_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))


モデルへの入力プロンプト（通常の生成）：



'<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。<|end|><|user|>新潟県へ旅行に行きます。おすすめを教えてください。<|end|><|assistant|>'


------------------------------------------------------------------------------------------------------------------------------------------------------

モデルへの入力プロンプト（RAG）：

<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。<|end|><|user|>新潟県へ旅行に行きます。おすすめを教えてください。

参考情報:Title: 【新潟県】観光スポットおすすめ22選｜最新から定番まで ...
Snippet: 2025年2月26日 · 米や米菓をテーマにしたテーマパークや酒蔵のほか、肥沃な平野や山間地帯が織り成す景観美が楽しめるスポットなど、具体的にどんな観光スポットがあるのか、エリア …

Title: 新潟の人気観光スポットおすすめ20選｜外せない定番・名所から ...
Snippet: 2024年9月19日 · そこで今回は、新潟県でおすすめの観光スポットを20ヶ所紹介します。 定番から穴場まで、またカップルや子連れなど目的に合わせた観光名所も紹介しているので、ぜひ …

Title: 【2025年保存版】新潟の穴場スポット20選！地元民おすすめの ...
Snippet: 2025年1月8日 · 新潟に旅行に行きたい人は必見の内容なので参考にしてくださいね。 米どころ、豪雪、日本酒などが有名な新潟県。 今回は、新潟市、長岡市、上越市と、エリアごとに厳 …

Title: 新潟県旅行完全ガイド！おすすめ観光スポット・グルメ・お ...
Snippet: 2024年10月28日 · 新潟県のおすすめ観光スポットや、気になるグルメ・お土産を事前にチェック！ おすすめの宿泊施設やパワースポット・絶景スポットまで、たくさんご紹介します！

Title: 新潟県のおすすめお出かけスポット・観光情報 - Gate to にいがた
Snippet: 2025年5月28日 · 新潟のおすすめお出かけスポットを探すなら「Gate to にいがた」。 定番観光地から穴場の絶景、スキー場、家族で遊べるレジャースポットまで、次の休日のヒン

もちろん！新潟県には魅力的な観光スポットがあります:

1. 新潟県立博物館: 地元の文化や歴史に関する展示が楽しめます。
2. 金沢湖: 美しい自然景観と水上活動が楽しめます。
3. 白根温泉: 温泉リゾートで、リラックスした休暇をお楽しみいただけます。
4. 新潟城: 日本の歴史的な城で、美しい庭園と展望台があります。
5. 佐渡島: 自然豊かな島で、ハイキングやボートツアーなどのアウトドア活動が楽しめます。

これらの場所がお役に立てば幸いです！


------------------------------------------------------------------------------------------------------------------------------------------------------

RAG時の出力：


新潟県には様々な観光スポットがあります。おすすめのスポットとしては:

1. 米どころ（米や米菓をテーマにしたテーマパークや酒蔵）
2. 豊かな自然景観を持つ肥沃な平野や山間地帯
3. 新潟市、長岡市、上越市などのエリアごとのおすすめスポット
4. スキー場やレジャースポットなど、家族で楽しめる場所

これらのスポットを訪れることで、新潟県の魅力を存分に体験できます。

### 検索するべき**"ではない"**入力；例文「こんにちは！」


RAGは，使うべき **"ではない"** 場面もあります．


2025年7月現在のブラウザ版ChatGPT等は，「ウェブで検索する」を選択せずとも，その入力文が【Web検索するべき文章か否か】を自動識別しています．

何故，全ての入力に対しWeb検索などをせず，識別する機能が必要なのか．


幾つか理由はありますが，その1つは検索するべき**ではない**入力も存在するためです．




In [ ]:

query = "こんにちは！"
k = 5

# DuckDuckGo 検索APIの検索結果を取得し
# 上位 k 件のスニペットをプロンプトに含めて回答を生成

ddgs = DDGS(headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=20)
results = list(ddgs.text(query, max_results=k,  region='ja-jp', safesearch='moderate', timelimit='y', backend='bing'))

# スニペットをプロンプト用に整形
context = "\n\n".join(
    f"Title: {item['title']}\nSnippet: {item['body']}"
    for item in results
)


messages_non_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。"},
    {"role": "user", "content":f"{query}"},
    ]

messages_rag = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。"},
    {"role": "user", "content":f"{query}\n\n参考情報:{context}"},
]

prompt_non_rag = processor.apply_chat_template(messages_non_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト（通常の生成）：\n")
display(prompt_non_rag)
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")



prompt_rag = processor.apply_chat_template(messages_rag, tokenize=False, add_generation_prompt=True)
print("モデルへの入力プロンプト（RAG）：\n")
print(prompt_rag)
print("\n"+ "".join([ "=" for n in range(150)]) + "\n")

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 256,
    "repetition_penalty": 1.05
}

outputs_non_rag = generator(prompt_non_rag, **generate_kwargs)
print("非RAG（通常の生成）時の出力：")
display(Markdown(outputs_non_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

outputs_rag = generator(prompt_rag, **generate_kwargs)
print("RAG時の出力：")
display(Markdown(outputs_rag[0]["generated_text"].split("<|assistant|>")[-1].strip()))


モデルへの入力プロンプト（通常の生成）：



'<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。<|end|><|user|>こんにちは！<|end|><|assistant|>'


------------------------------------------------------------------------------------------------------------------------------------------------------

モデルへの入力プロンプト（RAG）：

<|system|>あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。<|end|><|user|>こんにちは！

参考情報:Title: こんにちわ と こんにちは - 日本語 解決済 | 教えて!goo
Snippet: 2005年7月30日 · この質問は過去に何度か出されており、 多くの方の興味を弾くテーマなのだということが わかります。 こんにちは が歴史的には正しいです。  …

Title: こんにちわ、こんにちは、どっちが正しいの？ -メールの ...
Snippet: 2011年12月22日 · A No.6です。 質問者さまは混乱されているでしょうから整理します。 国語の決まりとしては「こんにちは」「今日は」です。 ただし最近は、「は …

Title: 「こんにちは」は何詞？ -日本語は、名詞や動詞、形容詞など ...
Snippet: 2005年4月15日 · 日本語は、名詞や動詞、形容詞などいろいろな品詞に分けられますが、「こんにちは」「こんばんは」などの挨拶語はどのような品詞に分けられる …

Title: 昔の人は「こんにちは」を何と言っていたのですか -日本語で ...
Snippet: 2011年8月5日 · 日本語であいさつは「こんにちは」ですが、これは挨拶としていつ頃から使われるようになったのでしょうか。また、「こんにちは」が使われるよう …

Title: 「付け合わせ」の意味を教えて下さい。 -こんにちは。会社で ...
Snippet: 2008年6月13日 · こんにちは。 会社で「書類を付け合せる」とたまに言いますが、実際日本語としてこれは正しいのでしょうか。 また、使い方として「A書類とB書 …<|end|><|assistant|>


非RAG（通常の生成）時の出力：


こんにちは！今日はどのようにお手伝いできますか？


------------------------------------------------------------------------------------------------------------------------------------------------------

RAG時の出力：


はい、「こんにちは」は挨拶として正しいです。「今日は」という表現も一般的ですが、「こんにちは」はより広く使用されています。両方とも日本語で適切に使用できます。

## ③質問文からキーワードを抽出して検索

In [ ]:

# 1) 質問文 から検索用のキーワードを抽出
# 2) DuckDuckGo で検索して上位 k 件のスニペットを取得
# 3) スニペットをプロンプトに含めて回答を生成


query = "明日の新潟市の天気は？"
max_keywords = 3
k = 3


messages_extract_keywords = [
    {"role": "system", "content": "あなたキーワード抽出が得意な自然言語処理モデルです。"},
    {"role": "user", "content":f"次の質問文から、検索エンジンで有効なキーワードを{max_keywords}語以内で抽出してください。"
    "キーワードのみを出力すること。\n\n"
    f"質問文：{query}\n\nキーワード：<|end|><|assistant|>"},
]


prompt_extract_keywords = processor.apply_chat_template(messages_extract_keywords, tokenize=False, add_generation_prompt=True)
print("キーワード抽出用のプロンプト：\n")
print(prompt_extract_keywords)
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

# キーワード抽出
generate_kwargs_extract_keywords = {
    "do_sample": True,
    "temperature": 0.01,
    "max_new_tokens": 16,
    "repetition_penalty": 1.05
}

outputs_extract_keywords = generator(prompt_extract_keywords, **generate_kwargs_extract_keywords)
keywords = outputs_extract_keywords[0]["generated_text"].split("<|assistant|>")[-1].strip()
print(f"抽出キーワード: {keywords}")
print("\n"+ "".join([ "=" for n in range(150)]) + "\n")


# DuckDuckGo 検索APIの検索結果を取得し
# 上位 k 件のスニペットをプロンプトに含めて回答を生成

ddgs = DDGS(headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=20)
results_keywords = list(ddgs.text(keywords, max_results=k,  region='ja-jp',
                         safesearch = 'moderate', timelimit='y', backend = 'bing'))

results_query = list(ddgs.text(query, max_results=k,  region='ja-jp',
                         safesearch = 'moderate', timelimit='y', backend = 'bing'))


# スニペットをプロンプト用に整形
context_keywords = "\n\n".join(
    f"Title: {item['title']}\nSnippet: {item['body']}"
    for item in results_keywords
)

context_query = "\n\n".join(
    f"Title: {item['title']}\nSnippet: {item['body']}"
    for item in results_query
)


messages_rag_keywords = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。"},
    {"role": "user", "content":f"{query}\n\n参考情報:{context_keywords}"},
]
messages_rag_query  = [
    {"role": "system", "content": "あなたはユーザからの質問へ簡潔に答える親切なAIアシスタントです。参考情報に基づいて、質問へ答えてください。"},
    {"role": "user", "content":f"{query}\n\n参考情報:{context_query}"},
]


prompt_rag_query = processor.apply_chat_template(messages_rag_query, tokenize=False, add_generation_prompt=True)
prompt_rag_keywords = processor.apply_chat_template(messages_rag_keywords, tokenize=False, add_generation_prompt=True)

print("モデルへの入力プロンプト（質問文でそのままWeb検索⇒RAG）：\n")
print(prompt_rag_query)
print("\n"+ "".join([ "-" for n in range(150)]) + "\n")

print("モデルへの入力プロンプト（キーワード抽出⇒Web検索⇒RAG）：\n")
print(prompt_rag_keywords)

print("\n"+ "".join([ "=" for n in range(150)]) + "\n")

generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 256,
    "repetition_penalty": 1.05
}

outputs_rag_query = generator(prompt_rag_query, **generate_kwargs)
print("質問文でそのままWeb検索⇒RAG時の出力：")
display(Markdown(outputs_rag_query[0]["generated_text"].split("<|assistant|>")[-1].strip()))

print("\n"+ "".join([ "-" for n in range(150)]) + "\n")


generate_kwargs = {
    "do_sample": True,
    "temperature": 0.000001,
    "max_new_tokens": 256,
    "repetition_penalty": 1.05
}

outputs_rag_keywords = generator(prompt_rag_keywords, **generate_kwargs)
print("キーワード抽出⇒Web検索⇒RAG時の出力：")
display(Markdown(outputs_rag_keywords[0]["generated_text"].split("<|assistant|>")[-1].strip()))
